# 1. Install all dependencies:

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Edit these before running (example of a valid config):


DATASET       = "mnist"       # mnist | cifar10 | cifar100 | fashionmnist | pathmnist | eurosat
NUM_CLIENTS   = 20
FRAC_CLIENTS  = 1.0
DIRICHLET_ALPHA = 0.5
TEST_SIZE     = 0.2
POISON_SIZE   = 0.35
ROUNDS        = 5
LOCAL_EPOCHS  = 3
BATCH_SIZE    = 32
CLIENT_LR     = 0.01
CLIENT_MOMENTUM = 0.9
WEIGHT_DECAY  = 1e-4
SERVER_OPT     = "fedopt"
SERVER_LR     = 0.7
ENABLE_FINGERPRINTING = "1"
FINGERPRINT_METHOD = "sparse" # choices: ["sparse", "dense"]
FINGERPRINT_SPARSITY = 0.01
TARGET_DOT_STRENGTH = 1.0
HONEST_FRACTION = 0.1
DETECTION_MARGIN = 1.5
SEED          = 42   
HISTORY_WINDOW = 5  
LABEL_FLIP_ALPHA = 1.0
BACKDOOR_TARGET_LABEL = 1
BACKDOOR_PATCH_SIZE = 15
BACKDOOR_INTENSITY = 1.0
TAU_BACKDOOR_THRESHOLD_STATISTICAL = 0.5
TAU_BACKDOOR_THRESHOLD_EMPRICAL = 0.5 
TARGETED_CLIENTS = "1"
VERIFIED_CLIENTS = "0 1 2"  
METHOD        = "label_flip"  # label_flip | backdoor | fingerprint
TARGETED      = [1, 2]        # adversarial client IDs
VERIFIED      = [0, 3]        # monitored client IDs
BUCKET_NAME   = "INSERT YOUR BUCKET NAME HERE"
BUCKET_FOLDER = "example_run"



In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "main.py",
    "--dataset_name",     DATASET,
    "--num_clients",      str(NUM_CLIENTS),
    "--rounds",           str(ROUNDS),
    "--local_epochs",     str(LOCAL_EPOCHS),
    "--method",           METHOD,
    "--targeted_clients", *[str(c) for c in TARGETED],
    "--verified_clients", *[str(c) for c in VERIFIED],
    "--bucket_name",      BUCKET_NAME,
    "--bucket_folder",    BUCKET_FOLDER,
], check=True)

# 3. Example run on sagemaker: 


In [ ]:

from sagemaker.huggingface import HuggingFace

role = "INSERT YOUR ROLE ARN HERE"

# Define hyperparameters that match your argparse arguments
hyperparameters = {
    # --- Dataset and client setup ---
    "dataset_name": DATASET,
    "num_clients": NUM_CLIENTS,
    "frac_clients": FRAC_CLIENTS,
    "dirichlet_alpha": DIRICHLET_ALPHA,
    "test_size":  TEST_SIZE,
    "poison_size": POISON_SIZE,

    # --- Training parameters ---
    "rounds": ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "batch_size": BATCH_SIZE, 
    "client_lr": CLIENT_LR,
    "client_momentum": CLIENT_MOMENTUM,
    "weight_decay": WEIGHT_DECAY,

    # --- Server optimisation ---
    "server_opt": SERVER_OPT,
    "server_lr": SERVER_LR,

    # --- Fingerprinting ---
    "enable_fingerprinting": ENABLE_FINGERPRINTING,          # or False if disabled
    "fingerprint_method": FINGERPRINT_METHOD,         
    "fingerprint_sparsity": FINGERPRINT_SPARSITY,
    "target_dot_strength": TARGET_DOT_STRENGTH,

    # --- Detection / Defence params ---
    "honest_fraction": HONEST_FRACTION,
    "detection_margin": DETECTION_MARGIN,
    "seed": SEED,
    "history_window": HISTORY_WINDOW,
    "method": METHOD,                 # choices: ["label_flip", "backdoor","fingerprint"]
    "label_flip_alpha": LABEL_FLIP_ALPHA,
    "backdoor_target_label": BACKDOOR_TARGET_LABEL,
    "backdoor_patch_size": BACKDOOR_PATCH_SIZE,
    "backdoor_intensity": BACKDOOR_INTENSITY,
    "tau_backdoor_threshold_statistical": TAU_BACKDOOR_THRESHOLD_STATISTICAL,
    "tau_backdoor_threshold_emprical": TAU_BACKDOOR_THRESHOLD_EMPRICAL,


    # --- Client selection ---
    # Lists must be passed as space-separated strings for SageMaker
    "targeted_clients": TARGETED_CLIENTS,
    "verified_clients": VERIFIED_CLIENTS,
}


huggingface_estimator = HuggingFace(
    entry_point="main.py",   # script inside source_dir
    source_dir="LINK TO THE SOURCE DIRECTORY IN S3",  # zipped dir
    role=role,
    instance_type="ml.g4dn.12xlarge",  # EXAMPLE  OF AN INSTANCE
    instance_count=1,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    base_job_name="JOB_NAME",
    hyperparameters=hyperparameters,  # pass args here
)

# Launch training
huggingface_estimator.fit(wait=False)
